Gold Layer + Business-Level Output

Drop tables if exists

In [0]:
spark.sql(f" drop table if exists gold.gold_analytics_table")
spark.sql(f" drop schema if exists gold.gold_analytics_table")

In [0]:
%sql
select count(*) from silver.fact_table

PySpark Transformations (Mandatory Techniques Used) 
Aggregations Grouping Window functions

Episodes per Season (Aggregation + Grouping)

In [0]:
fact_df   = spark.table("silver.fact_table")

In [0]:
from pyspark.sql.functions import count

episodes_per_season_df = (
    fact_df
    .groupBy("id", "season")
    .agg(count("episode_id").alias("episodes_per_season"))
)


In [0]:
episodes_per_season_df.display()

2.Average Runtime per Show

In [0]:
df_shows   = spark.table("silver.shows")

In [0]:
from pyspark.sql.functions import avg

avg_runtime_df = (
    df_shows
    .groupBy("id", "name", "genres")
    .agg(avg("runtime").alias("avg_runtime"))
)

In [0]:
avg_runtime_df.display()

3 Top Cast Members (Window Function)
Count appearances

In [0]:
df_cast   = spark.table("silver.cast")

In [0]:

from pyspark.sql.functions import count

cast_count_df = (
    df_cast
    .groupBy("show_id", "person_name")
    .agg(count("person_id").alias("appearances"))
)

In [0]:
cast_count_df.display() 

2: Rank actors per show

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank, col

cast_window = Window.partitionBy("show_id").orderBy(col("appearances").desc())

top_cast_df = (
    cast_count_df
    .withColumn("id", col("show_id"))
    .withColumn("cast_rank", dense_rank().over(cast_window))
    .filter(col("cast_rank") <= 3)
)

In [0]:
top_cast_df.display()

4 Most Common Genres (Window + Aggregation)

In [0]:

genre_count_df = (
    df_shows
    .groupBy("genres")
    .agg(count("id").alias("genre_count"))
)

genre_window = Window.orderBy(col("genre_count").desc())

top_genres_df = (
    genre_count_df
    .withColumn("genre_rank", dense_rank().over(genre_window))
)


In [0]:
genre_count_df.display()

In [0]:
top_genres_df.display()

Create the Final Gold Fact Table

In [0]:
gold_fact_df = (
    episodes_per_season_df
    .join(avg_runtime_df, "id", "inner")
    .join(top_cast_df, "id", "left")
    .join(top_genres_df, "genres", "left")
    .select(
        "id",
        col("name").alias("show_title"),
        "genres",
        "season",
        "episodes_per_season",
        "avg_runtime",
        "person_name",
        "cast_rank",
        "genre_rank"
    )
)

In [0]:
gold_fact_df.display()

Deliverable :Write Gold Delta Table

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

In [0]:
gold_fact_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("mergeSchema", "true")\
    .saveAsTable("gold.gold_analytics_table")

In [0]:
gold_fact_df.display()  

Deliverable : Databricks SQL Queries

Episodes per Season

In [0]:
%sql

SELECT show_title, season, episodes_per_season
FROM gold.gold_analytics_table
ORDER BY show_title, season;

Average Runtime per Show

In [0]:
%sql

SELECT show_title, avg_runtime
FROM gold.gold_analytics_table
GROUP BY show_title, avg_runtime
ORDER BY avg_runtime DESC;

Top Cast Members per Show

In [0]:
%sql

SELECT show_title, person_name, cast_rank
FROM gold.gold_analytics_table
WHERE cast_rank <= 3
ORDER BY show_title, cast_rank;

Most Common Genres

In [0]:
%sql

SELECT genres, genre_rank
FROM gold.gold_analytics_table
ORDER BY genre_rank DESC;